In [ ]:
import numpy
import random
import uuid
from algorithms.models import RPDHGParams, SolverConfig, GurobiParams
from algorithms.pdhg_restarted_cu import pdhg_restarted_cu
from algorithms.gurobi_lp import solve_lp_gurobi
from experiments.evaluation import evaluate_solver
from pathlib import Path
from dataclasses import replace
import transportation_problems
from transportation_problems.dotmark.loader_csv import load_dotmark_instance_csv
from transportation_problems.dotmark.build_ot_problem import build_ot_lp

def get_string_tuple_list(folder:str):
    string_Y = "transportation_problems/dotmark/csv_data/FOLDER/data32_XXXX.csv"
    string_X = string_Y.replace("FOLDER", folder)
    
    number_tuple_list = []
    for i in range(1,11):
        for j in range(i +1,11):
            number_tuple_list.append((i +1000, j+1000))
    
    string_tuple_list = []
    for tpl in number_tuple_list:
        string_A = string_X.replace("XXXX", str(tpl[0]))
        string_D = string_X.replace("XXXX", str(tpl[1]))
        string_tuple_list.append((string_A, string_D))
        
    return string_tuple_list

folder_names = ["LogGRF","MicroscopyImages","WhiteNoise","CauchyDensity","ClassicImages","GRFmoderate","GRFrough","GRFsmooth","LogitGRF","Shapes"]

for folder_name in folder_names:

    for x in get_string_tuple_list(folder_name):
        path_A = Path(x[0])
        path_B = Path(x[1])

        print("Loading images...")
        Aimg, Bimg, a, b = load_dotmark_instance_csv(path_A, path_B)
        H, W = Aimg.shape

        print("Building OT LP...")

        from_number = x[0].split("/").pop()
        to_number = x[1].split("/").pop()

        good_name = folder_name + str(from_number) + "_to_" + str(to_number)

        dotmark_problem = build_ot_lp(a, b, H, W, name = good_name)

        print("LP size:", dotmark_problem.A.shape)


        tolerance = 1e-8
        my_params1 = RPDHGParams(
                        tau = None,
                        sigma = None,
                        theta = 1.03,
                        alpha= 1.0,
                        rebalancing_threshhold = 3.0,
                        step_shrinkage = 0.75,
                        restart_check = "adaptive" ,   #"adaptive" | "fixed" | "none"
                        min_epoch_length = 250,
                        max_iter = 100_000,
                        fixed_iter_restart= 3000,
                        tol_primal = tolerance,
                        tol_dual = tolerance,
                        tol_gap = tolerance,
                        tau_sigma_preconditioned= True,
                        rebalance_tau_sigma= True,
                        diagnostik_i = 25
        )


        directory_name = r"experiments\32 none"

        """
        gurobi_params = GurobiParams()
        evaluate_solver(
                problem=dotmark_problem,
                solver_fn= lambda dotmark_problem: solve_lp_gurobi(dotmark_problem,gurobi_params),
                write_run=True,
                csv_export= False,
                exp_dir=directory_name,
                experiment_name=str(uuid.uuid4()),
                solver_config= SolverConfig(solver_name= "gurobi", params = gurobi_params)
            )"""


        my_params2 = replace(my_params1,restart_check="none")
        params = [my_params2]
        for p in params:

            evaluate_solver(
                problem=dotmark_problem,
                solver_fn= lambda dotmark_problem: pdhg_restarted_cu(dotmark_problem,p),
                write_run=True,
                csv_export= False,
                exp_dir=directory_name,
                experiment_name=str(uuid.uuid4()),
                solver_config= SolverConfig(solver_name= "noRestart", params = p)
            )



C:\Users\Felix\PycharmProjects\rPDHGprivate\.venv\Lib\site-packages\cupy\_environment.py:275: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


Loading images...
Building OT LP...
LP size: (2048, 1048576)
Iter 25 | Primal: 1.8167e+00 | relPRes: 2.70e-01 | relDRes: 9.00e-04 | relGap: 5.52e-01|restartcheck: none | tau: 2.1709385485438393
Iter 50 | Primal: 3.8610e+00 | relPRes: 1.38e-01 | relDRes: 3.55e-04 | relGap: 4.09e-01|restartcheck: none | tau: 4.545463219576649
Iter 75 | Primal: 7.6906e+00 | relPRes: 5.57e-02 | relDRes: 1.95e-04 | relGap: 1.47e-01|restartcheck: none | tau: 9.517190569204582
Iter 100 | Primal: 9.2380e+00 | relPRes: 3.35e-02 | relDRes: 1.20e-04 | relGap: 6.21e-02|restartcheck: none | tau: 7.924077674101478
Iter 125 | Primal: 9.7549e+00 | relPRes: 2.27e-02 | relDRes: 8.60e-05 | relGap: 3.65e-02|restartcheck: none | tau: 6.597641029525102
Iter 150 | Primal: 9.8642e+00 | relPRes: 1.54e-02 | relDRes: 6.68e-05 | relGap: 2.97e-02|restartcheck: none | tau: 13.813995175400786
Iter 175 | Primal: 9.9247e+00 | relPRes: 1.09e-02 | relDRes: 5.73e-05 | relGap: 2.53e-02|restartcheck: none | tau: 11.501626447802389
Iter 200